## RSNA Pneumonia Detection Challenge Data Cleaning
<p>The RSNA Pneumonia Detection Challenge is a very old practice competition on Kaggle. If one wants to perform this with yolo, one would require to make certain changes to the data to prepare it as yaml for yolo. This notebook presents a way to change dicom to jpeg and csv to txt and further copying them to respective yaml structured folders</p>

### Install dependencies
Install them as a rule

In [1]:
%%capture
import os
! pip install scikit-learn numpy matplotlib pandas lightning torch timm==0.5.4
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
import lightning as L
from torchvision import datasets, transforms
import pandas as pd
import numpy as np
from glob import glob
from PIL import Image
import logging
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

### Label Change
One has to change the labels in the form of x,y,w,h. They are always normalized, so whichever value you give as size, it would not matter.

In [2]:
label_src="/kaggle/input/competitions/rsna-pneumonia-detection-challenge/stage_2_train_labels.csv"
out="label"
IMG_W = 1024
IMG_H = 1024
os.makedirs(out,exist_ok=True)
df=pd.read_csv(label_src)
for patientId,group in df.groupby("patientId"):
    label_path=os.path.join(out, f"{patientId}.txt")
    with open(label_path,"w") as f:
        for _, row in group.iterrows():
            if row["Target"] == 0:
                continue

            x = row["x"]
            y = row["y"]
            w = row["width"]
            h = row["height"]
            
            x_center=(x + w/2)/IMG_W
            y_center=(y + h/2)/IMG_H
            w_norm=w/IMG_W
            h_norm=h/IMG_H
            f.write(f"0 {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}\n")


### Image Change
Change the images from DICOM to JPEG. Yolo needs images in jpeg

In [3]:
import pydicom
import cv2

DICOM_DIR="/kaggle/input/competitions/rsna-pneumonia-detection-challenge/stage_2_train_images"
OUTPUT_DIR="/kaggle/working/rsna_yolo/images/train"

os.makedirs(OUTPUT_DIR,exist_ok=True)

for file in os.listdir(DICOM_DIR):
    if not file.endswith(".dcm"):
        continue

    dcm_path = os.path.join(DICOM_DIR, file)

    ds = pydicom.dcmread(dcm_path)
    img = ds.pixel_array.astype(np.float32)

    if hasattr(ds, "PhotometricInterpretation"):
        if ds.PhotometricInterpretation == "MONOCHROME1":
            img = np.max(img) - img
    img = img - np.min(img)
    if np.max(img) != 0:
        img = img / np.max(img)
    img = (img * 255).astype(np.uint8)
    filename = file.replace(".dcm", ".png")
    save_path = os.path.join(OUTPUT_DIR, filename)
    cv2.imwrite(save_path, img)

print(f"Done. PNG images saved in: {OUTPUT_DIR}")

Done. PNG images saved in: /kaggle/working/rsna_yolo/images/train


### YOLO'S YAML FORMAT
Next you  can reorder your files in this format:

```yaml
rsna_yolo/
│
├── images/
│   ├── train/
│   └── val/
│
├── labels/
│   ├── train/
│   └── val/
│
└── data.yaml
```

Because YOLO needs it this way.
Use this block of code for the purpose:

```python
import shutil # Move files instead of copying
import random

IMAGE_DIR = "/kaggle/working/rsna_yolo/images"
LABEL_DIR = "/kaggle/working/label"

BASE_DIR = "/kaggle/working/rsna_yolo"

TRAIN_IMG_DIR = os.path.join(BASE_DIR, "images/train")
VAL_IMG_DIR = os.path.join(BASE_DIR, "images/val")

TRAIN_LABEL_DIR = os.path.join(BASE_DIR, "labels/train")
VAL_LABEL_DIR = os.path.join(BASE_DIR, "labels/val")

for path in [
    TRAIN_IMG_DIR,
    VAL_IMG_DIR,
    TRAIN_LABEL_DIR,
    VAL_LABEL_DIR
]:
    os.makedirs(path, exist_ok=True)

images = [
    f for f in os.listdir(IMAGE_DIR)
    if f.endswith(".png")
]

random.seed(42)
random.shuffle(images)

# 70-30 split
split_idx = int(0.7 * len(images))

train_images = images[:split_idx]
val_images = images[split_idx:]

def move_files(file_list, img_dest, label_dest):
    for img_file in file_list:
        base_name = os.path.splitext(img_file)[0]
        label_file = base_name + ".txt"

        src_img = os.path.join(IMAGE_DIR, img_file)
        src_label = os.path.join(LABEL_DIR, label_file)

        dst_img = os.path.join(img_dest, img_file)
        dst_label = os.path.join(label_dest, label_file)

        if os.path.exists(src_img):
            shutil.move(src_img, dst_img)

        if os.path.exists(src_label):
            shutil.move(src_label, dst_label)
        else:
            open(dst_label, "w").close()

move_files(train_images, TRAIN_IMG_DIR, TRAIN_LABEL_DIR)
move_files(val_images, VAL_IMG_DIR, VAL_LABEL_DIR)

print("Done.")
print(f"Train images: {len(train_images)}")
print(f"Val images: {len(val_images)}")
```